# Notebook 03 — Evaluate All Experiments
**Member: C** | Set `model.checkpoint_path` in `configs/eval.yaml` before each run.

| Experiment | `checkpoint_path` | Benchmark |
|---|---|---|
| Exp 0 zero-shot    | `null`                             | GSM8K |
| Exp 1 few-shot CoT | `null`                             | GSM8K |
| Exp 2 adaptation   | `results/checkpoints/adaptation`   | GSM8K |
| Exp 3 distillation | `results/checkpoints/distillation` | GSM8K |
| Exp 4 filtered     | `results/checkpoints/filtered`     | GSM8K |
| Exp 5 cross-task   | `results/checkpoints/filtered`     | ARC |

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

repo_url = "https://github.com/TIFFMUN/SC4001-Group133.git"

if not os.path.exists("/content/SC4001-Group133/.git"):
    os.system(f"git clone {repo_url} /content/SC4001-Group133")

os.chdir("/content/SC4001-Group133")
os.system("git pull origin main")

!pip install -r requirements.txt -q
!pip install -e . -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sc4001-group133 (pyproject.toml) ... done


In [5]:
# Redirect HF cache + data/ + results/ to Drive — run before any HF imports
import importlib
importlib.invalidate_caches()
from src.setup_cache import setup
setup()

[setup_cache] Cache redirected to Drive: /content/drive/MyDrive/SC4001


EXP 3

In [6]:
from pathlib import Path

root = Path("/content/drive/MyDrive/SC4001")
checkpoint = root / "updated_data/checkpoints/distillation_updated"

print("Drive mounted:", Path("/content/drive/MyDrive").exists())
print("SC4001 folder exists:", root.exists())
print("Checkpoint folder exists:", checkpoint.exists())

if checkpoint.exists():
    print("\nFiles in checkpoint:")
    for file in checkpoint.iterdir():
        print(file.name)

print("\nSearching for Experiment 3 adapter...")
for file in root.rglob("adapter_model.safetensors"):
    if "distillation" in str(file).lower():
        print("FOUND:", file)

Drive mounted: True
SC4001 folder exists: True
Checkpoint folder exists: False

Searching for Experiment 3 adapter...
FOUND: /content/drive/MyDrive/SC4001/data/checkpoints/distillation_updated/adapter_model.safetensors
FOUND: /content/drive/MyDrive/SC4001/data/checkpoints/distillation_updated/checkpoint-90/adapter_model.safetensors
FOUND: /content/drive/MyDrive/SC4001/results/checkpoints/distillation/adapter_model.safetensors
FOUND: /content/drive/MyDrive/SC4001/results/checkpoints/distillation/checkpoint-18/adapter_model.safetensors


In [18]:
# Experiment 3 — Full GSM8K evaluation
# 500 test questions, 600 maximum output tokens

import yaml
from pathlib import Path
from src.evaluation.eval_gsm8k import evaluate_gsm8k

config_path = Path("configs/eval.yaml")

with config_path.open() as f:
    cfg = yaml.safe_load(f)

# Updated Experiment 3 checkpoint
checkpoint = Path(
    "/content/drive/MyDrive/SC4001/data/checkpoints/distillation_updated"
)

assert (checkpoint / "adapter_model.safetensors").is_file()
assert (checkpoint / "adapter_config.json").is_file()

# Evaluation settings
cfg["model"]["checkpoint_path"] = str(checkpoint)
cfg["generation"]["max_new_tokens"] = 600
cfg["generation"]["do_sample"] = False
cfg["gsm8k"]["max_samples"] = 500

assert cfg["gsm8k"]["split"] == "test"

with config_path.open("w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

EXPERIMENT_NAME = "exp3_distillation_updated_500_600tokens"

print("Experiment:", EXPERIMENT_NAME)
print("Checkpoint:", checkpoint)
print("Test samples:", 500)
print("Max output tokens:", 600)

evaluate_gsm8k(
    experiment_name=EXPERIMENT_NAME,
    max_samples=500
)

Experiment: exp3_distillation_updated_500_600tokens
Checkpoint: /content/drive/MyDrive/SC4001/data/checkpoints/distillation_updated
Test samples: 500
Max output tokens: 600


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Evaluating GSM8K [exp3_distillation_updated_500_600tokens]: 100%|██████████| 500/500 [1:37:10<00:00, 11.66s/it]


[{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
  'ground_truth': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18',
  'prediction': "To find out how much money Janet makes every day at the farmers' market, we need to calculate the total number of eggs laid by the ducks and then determine the revenue from selling those eggs.\n\n**Step 1: Calculate the total number of eggs laid per day.**\n*   **Ducks:** The problem states that the ducks lay 16 eggs per day.\n*   **Breakfast:** Janet eats 3 eggs (breakfast) per day.\n*   **Baking:** She bakes muffins for her friends, which means she also bakes muffins for herself. Since the problem doesn't specify if she bake

EXP 4

In [6]:
from pathlib import Path

root = Path("/content/drive/MyDrive/SC4001")

for file in root.rglob("adapter_model.safetensors"):
    if "filtered_updated" in str(file):
        print("FOUND:", file.parent)

FOUND: /content/drive/MyDrive/SC4001/data/checkpoints/filtered_updated
FOUND: /content/drive/MyDrive/SC4001/data/checkpoints/filtered_updated/checkpoint-69


In [ ]:
# Experiment 4 — Full GSM8K evaluation
# 500 test questions, 600 maximum output tokens

import yaml
from pathlib import Path
from src.evaluation.eval_gsm8k import evaluate_gsm8k

config_path = Path("configs/eval.yaml")

with config_path.open() as f:
    cfg = yaml.safe_load(f)

# Updated Experiment 4 checkpoint
checkpoint = Path(
    "/content/drive/MyDrive/SC4001/data/checkpoints/filtered_updated"
)

assert (checkpoint / "adapter_model.safetensors").is_file()
assert (checkpoint / "adapter_config.json").is_file()

# Evaluation settings
cfg["model"]["checkpoint_path"] = str(checkpoint)
cfg["generation"]["max_new_tokens"] = 600
cfg["generation"]["do_sample"] = False
cfg["gsm8k"]["max_samples"] = 500

assert cfg["gsm8k"]["split"] == "test"

with config_path.open("w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

EXPERIMENT_NAME = "exp4_filtered_updated_500_600tokens"

print("Experiment:", EXPERIMENT_NAME)
print("Checkpoint:", checkpoint)
print("Test samples:", 500)
print("Max output tokens:", 600)

evaluate_gsm8k(
    experiment_name=EXPERIMENT_NAME,
    max_samples=500
)

W1115 21:32:56.390000 1013 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1115 21:32:56.439000 1013 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Experiment: exp4_filtered_updated_500_600tokens
Checkpoint: /content/drive/MyDrive/SC4001/data/checkpoints/filtered_updated
Test samples: 500
Max output tokens: 600


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Evaluating GSM8K [exp4_filtered_updated_500_600tokens]:   3%|▎         | 15/500 [20:46<12:46:23, 94.81s/it]

In [ ]:
# ARC-Challenge — Experiment 3 (Standard Distillation)
import yaml
from pathlib import Path
from src.evaluation.eval_arc import evaluate_arc

config_path = Path("configs/eval.yaml")

with config_path.open() as f:
    cfg = yaml.safe_load(f)

checkpoint = Path(
    "/content/drive/MyDrive/SC4001/data/checkpoints/distillation_updated"
)

assert (checkpoint / "adapter_model.safetensors").is_file()
assert (checkpoint / "adapter_config.json").is_file()

cfg["model"]["checkpoint_path"] = str(checkpoint)

cfg["arc"]["dataset"] = "allenai/ai2_arc"
cfg["arc"]["config"] = "ARC-Challenge"
cfg["arc"]["split"] = "test"
cfg["arc"]["max_samples"] = None  # Full test set

cfg["generation"]["max_new_tokens"] = 600
cfg["generation"]["do_sample"] = False

with config_path.open("w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Evaluating Experiment 3 on ARC-Challenge")
print("Checkpoint:", checkpoint)

evaluate_arc(
    experiment_name="exp5_arc_exp3_updated_600tokens"
)

In [ ]:
# ARC-Challenge — Experiment 4 (Filtered Distillation)
import yaml
from pathlib import Path
from src.evaluation.eval_arc import evaluate_arc

config_path = Path("configs/eval.yaml")

with config_path.open() as f:
    cfg = yaml.safe_load(f)

checkpoint = Path(
    "/content/drive/MyDrive/SC4001/data/checkpoints/filtered_updated"
)

assert (checkpoint / "adapter_model.safetensors").is_file()
assert (checkpoint / "adapter_config.json").is_file()

cfg["model"]["checkpoint_path"] = str(checkpoint)

cfg["arc"]["dataset"] = "allenai/ai2_arc"
cfg["arc"]["config"] = "ARC-Challenge"
cfg["arc"]["split"] = "test"
cfg["arc"]["max_samples"] = None  # Full test set

cfg["generation"]["max_new_tokens"] = 600
cfg["generation"]["do_sample"] = False

with config_path.open("w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Evaluating Experiment 4 on ARC-Challenge")
print("Checkpoint:", checkpoint)

evaluate_arc(
    experiment_name="exp5_arc_exp4_updated_600tokens"
)

In [ ]:
# Exp 2-4 — set checkpoint_path in configs/eval.yaml, update EXPERIMENT_NAME, then run
EXPERIMENT_NAME = 'exp2_adaptation_r16'  # change per run
MAX_SAMPLES = 5  # set to 5 for quick test, 100 for full eval, None for all 1319

from src.evaluation.eval_gsm8k import evaluate_gsm8k
evaluate_gsm8k(experiment_name=EXPERIMENT_NAME, max_samples=MAX_SAMPLES)

In [ ]:
# Exp 1 — few-shot CoT baseline
MAX_SAMPLES = 5  # set to 5 for quick test, 100 for full eval, None for all 1319

results = run_baseline(mode='few_shot', dataset_name='gsm8k', max_samples=MAX_SAMPLES)

for r in results:
    r['is_correct'] = answers_match(extract_answer(r['prediction']), r['ground_truth'])

save_results(results, 'results/exp1_few_shot_gsm8k.csv')
print(compute_summary_stats(results))

In [ ]:
# Prompt ablation — test all 4 new prompt variants on a small sample
MAX_SAMPLES = 5  # increase to 100 for real results

import json
from pathlib import Path
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from tqdm import tqdm

from src.baselines.cot_prompting import (
    build_one_shot_prompt,
    build_zero_shot_cot_prompt,
    build_concise_prompt,
    build_format_reinforced_prompt,
)
from src.distillation.filter_traces import extract_answer, answers_match
from src.evaluation.metrics import compute_latency, compute_summary_stats, save_results, print_summary

# Load model once, reuse for all variants
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model.eval()

test_data = load_dataset("openai/gsm8k", "main")["test"].select(range(MAX_SAMPLES))

prompt_variants = {
    "exp1b_one_shot":           build_one_shot_prompt,
    "exp1c_zs_cot":             build_zero_shot_cot_prompt,
    "exp1d_concise":            build_concise_prompt,
    "exp1e_format_reinforced":  build_format_reinforced_prompt,
}

all_stats = {}
for name, prompt_fn in prompt_variants.items():
    results = []
    with torch.no_grad():
        for sample in tqdm(test_data, desc=name):
            prompt = prompt_fn(sample["question"], tokenizer)
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            input_len = inputs["input_ids"].shape[1]
            latency_ms, output_ids = compute_latency(
                model=model, inputs=inputs, max_new_tokens=256, do_sample=False
            )
            generated_ids = output_ids[0][input_len:]
            prediction = tokenizer.decode(generated_ids, skip_special_tokens=True)
            predicted_answer = extract_answer(prediction)
            is_correct = answers_match(predicted_answer, sample["answer"])
            results.append({
                "question": sample["question"],
                "ground_truth": sample["answer"],
                "prediction": prediction,
                "predicted_answer": predicted_answer,
                "is_correct": is_correct,
                "latency_ms": latency_ms,
                "token_count": len(generated_ids),
            })
    stats = compute_summary_stats(results)
    print_summary(name, stats)
    save_results(results, f"results/{name}_gsm8k.csv")

    # Save summary stats JSON (same as evaluate_gsm8k does)
    stats_path = Path("results/summary_results") / f"{name}_gsm8k_summary.json"
    stats_path.parent.mkdir(parents=True, exist_ok=True)
    with open(stats_path, "w") as f:
        json.dump({"experiment": name, **stats}, f, indent=2)

    all_stats[name] = stats

# Summary table
print("\n=== COMPARISON ===")
print(f"{'Variant':<30} {'Accuracy':>10} {'Mean Tokens':>12}")
print("-" * 55)
print(f"{'exp0_zero_shot (baseline)':<30} {'29.00%':>10} {'241.8':>12}")
for name, s in all_stats.items():
    print(f"{name:<30} {s['accuracy']:>10.2%} {s['mean_token_count']:>12.1f}")

In [ ]:
# Exp 0 — zero-shot baseline
MAX_SAMPLES = 5  # set to 5 for quick test, 100 for full eval, None for all 1319

from src.baselines.cot_prompting import run_baseline
from src.distillation.filter_traces import extract_answer, answers_match
from src.evaluation.metrics import compute_summary_stats, save_results

results = run_baseline(mode='zero_shot', dataset_name='gsm8k', max_samples=MAX_SAMPLES)

for r in results:
    r['is_correct'] = answers_match(extract_answer(r['prediction']), r['ground_truth'])

save_results(results, 'results/exp0_zero_shot_gsm8k.csv')
print(compute_summary_stats(results))

In [ ]:
# List all saved CSVs on Drive
csvs = [f for f in os.listdir('/content/drive/MyDrive/SC4001/results') if f.endswith('.csv')]
for f in sorted(csvs):
    print(f)

## Exp 6 — Inference-gated budget router (optional, run after Exps 0–5)

Prerequisite: set `router.enabled: true` and `router.checkpoint_path` in `configs/eval.yaml`.

The router sweeps a range of complexity thresholds and saves a Pareto-ready CSV to
`results/exp6_router_sweep.csv`. Use `max_samples=50` for a quick sanity check first.

In [ ]:
# Exp 6 — budget router threshold sweep
# Step 1: quick sanity check with 50 samples
from src.evaluation.eval_router import run_router_sweep

sweep_df = run_router_sweep(experiment_name='exp6_router', max_samples=50)
print(sweep_df[['threshold', 'accuracy', 'mean_token_count', 'pct_easy']].to_string(index=False))

In [ ]:
# Exp 6 — full sweep (set max_samples: null in eval.yaml for full 1319-sample eval)
# After confirming the sanity check above looks reasonable, run:
sweep_df_full = run_router_sweep(experiment_name='exp6_router_full', max_samples=None)
print(sweep_df_full[['threshold', 'accuracy', 'mean_token_count', 'pct_easy']].to_string(index=False))